# Phase 8 — Mean-CVaR Risk Parity Allocation

**Objective:** Construct an alternative dynamic allocation using CVaR risk measure
(Rockafellar-Uryasev LP) with regime-based tilts. Benchmark against the
Black-Litterman weights from Phase 7.

**Sections:**
1. Setup & Load
2. Risk Parity Base Weights
3. Regime Tilt
4. CVaR Optimization (Rockafellar-Uryasev LP)
5. Walk-Forward CVaR Weight Time Series
6. Lambda Sensitivity Analysis
7. Comparison with BL

**Inputs:** `factor_returns_full.parquet`, `regime_probabilities.parquet`,
`garch_conditional_vol.parquet`, `conditional_covariance.parquet`,
`bl_weights_timeseries.parquet`

**Outputs:** `cvar_weights_timeseries.parquet`, figures, comparison table

---
## 8.1 Setup & Load

In [1]:
import sys
import os
import logging
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src is importable
sys.path.insert(0, os.path.abspath('..'))

from src.config import (
    PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR, COLORS,
    FACTOR_RETURNS_FULL_FILE, REGIME_PROBS_FILE, COND_VOL_FILE,
    COND_COV_FILE, BL_WEIGHTS_FILE, CVAR_WEIGHTS_FILE,
    MAX_WEIGHT_FACTOR, TURNOVER_LIMIT, TC_FACTOR, RANDOM_STATE,
    FACTOR_LIVE_START,
)
from src.validation import validate_parquet, check_nan_propagation
from src.visualization import setup_style, save_fig, plot_weight_evolution
from src.portfolio_optimization import (
    mean_cvar_optimize, risk_budgeting_optimize,
    compare_allocations, risk_decomposition, concentration_metrics,
)
from src.risk_metrics import portfolio_cvar, compute_all_metrics
from src.regime_model import regime_conditional_stats

# Logging
PHASE_NUM = 8
log_file = LOGS_DIR / f'phase_{PHASE_NUM}_{datetime.now():%Y%m%d_%H%M}.log'
logging.basicConfig(
    filename=str(log_file),
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
)
logger = logging.getLogger(__name__)
logger.info(f'Phase {PHASE_NUM} started')

warnings.filterwarnings('ignore', category=FutureWarning)
setup_style()
np.random.seed(RANDOM_STATE)

# Constants
FACTORS = ['hml', 'umd', 'rmw', 'lowvol']
WEIGHT_COLS = ['w_hml', 'w_umd', 'w_rmw', 'w_lowvol']
ALPHA_CVAR = 0.05          # 5% tail for CVaR
WARMUP_MONTHS = 60         # Minimum expanding-window months

print(f'Phase {PHASE_NUM}: Mean-CVaR Risk Parity Allocation')
print(f'Log file: {log_file}')

Phase 8: Mean-CVaR Risk Parity Allocation
Log file: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/logs/phase_8_20260328_0258.log


In [2]:
# ── Load all input data ──────────────────────────────────────────────────────

factor_returns = pd.read_parquet(FACTOR_RETURNS_FULL_FILE)
validate_parquet(factor_returns, expected_cols=FACTORS, min_rows=100,
                 date_index=True, label='factor_returns_full')

regime_probs = pd.read_parquet(REGIME_PROBS_FILE)
validate_parquet(regime_probs,
                 expected_cols=['p_expansion', 'p_slowdown', 'p_crisis'],
                 min_rows=100, date_index=True, label='regime_probabilities')

garch_vol = pd.read_parquet(COND_VOL_FILE)
vol_cols = ['vol_hml', 'vol_umd', 'vol_rmw', 'vol_lowvol']
validate_parquet(garch_vol, expected_cols=vol_cols,
                 min_rows=100, date_index=True, label='garch_conditional_vol')

cond_cov = pd.read_parquet(COND_COV_FILE)
validate_parquet(cond_cov, min_rows=100, date_index=True,
                 label='conditional_covariance')

print(f'factor_returns: {factor_returns.shape}')
print(f'regime_probs:   {regime_probs.shape}')
print(f'garch_vol:      {garch_vol.shape}')
print(f'cond_cov:       {cond_cov.shape}')

factor_returns: (232, 4)
regime_probs:   (186, 4)
garch_vol:      (232, 4)
cond_cov:       (231, 16)


In [3]:
# ── Align dates (inner join, month-end) ──────────────────────────────────────

# Normalise all indices to month-end
for df in [factor_returns, regime_probs, garch_vol, cond_cov]:
    df.index = df.index + pd.offsets.MonthEnd(0)

# Inner join on common dates
common_idx = (
    factor_returns.index
    .intersection(regime_probs.index)
    .intersection(garch_vol.index)
    .intersection(cond_cov.index)
)
common_idx = common_idx.sort_values().unique()

factor_returns = factor_returns.loc[common_idx, FACTORS]
regime_probs = regime_probs.loc[common_idx]
garch_vol = garch_vol.loc[common_idx, vol_cols]
cond_cov = cond_cov.loc[common_idx]

assert factor_returns.index.is_unique, 'Duplicate dates in factor_returns'
assert factor_returns.index.equals(regime_probs.index), 'Index mismatch'

print(f'Aligned date range: {common_idx[0].date()} to {common_idx[-1].date()}')
print(f'Total months: {len(common_idx)}')

Aligned date range: 2008-11-30 to 2024-03-31
Total months: 185


---
## 8.2 Risk Parity Base Weights

$$w_{RP,i,t} = \frac{1/\sigma_{i,t}}{\sum_j 1/\sigma_{j,t}}$$

Using GARCH conditional volatility at each month-end.

In [4]:
# Compute inverse-vol risk parity weights at each t
inv_vol = 1.0 / garch_vol.values  # (T, 4)
rp_weights = inv_vol / inv_vol.sum(axis=1, keepdims=True)  # normalise to sum=1

rp_weights_df = pd.DataFrame(
    rp_weights, index=garch_vol.index, columns=WEIGHT_COLS
)

# Sanity checks
assert np.allclose(rp_weights_df.sum(axis=1), 1.0, atol=1e-8), 'RP weights do not sum to 1'
assert (rp_weights_df >= 0).all().all(), 'Negative RP weights'

print('Risk Parity base weights — summary statistics:')
print(rp_weights_df.describe().round(4))

Risk Parity base weights — summary statistics:
          w_hml     w_umd     w_rmw  w_lowvol
count  185.0000  185.0000  185.0000  185.0000
mean     0.2461    0.2070    0.4098    0.1371
std      0.0545    0.0603    0.0737    0.0269
min      0.1294    0.0528    0.2693    0.0765
25%      0.2039    0.1613    0.3584    0.1184
50%      0.2422    0.2091    0.3902    0.1393
75%      0.2818    0.2530    0.4546    0.1561
max      0.4040    0.3466    0.6384    0.1950


In [5]:
# Plot risk parity weight evolution
fig = plot_weight_evolution(
    rp_weights_df.rename(columns=dict(zip(WEIGHT_COLS, FACTORS))),
    title='Risk Parity Base Weights (Inverse GARCH Vol)'
)
save_fig(fig, 'cvar_rp_base_weights')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_rp_base_weights.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_rp_base_weights.png')

---
## 8.3 Regime Tilt

$$\text{tilt}_{i,t} = 1 + \lambda \left( \sum_k p_{k,t} \cdot SR_{i,k} - \overline{SR}_i \right)$$

where $SR_{i,k}$ is the Sharpe ratio of factor $i$ in regime $k$, and $\overline{SR}_i$
is the unconditional Sharpe ratio of factor $i$.

- Clip tilts to $[0.5, 2.0]$.
- Test $\lambda \in \{0.5, 1.0, 1.5, 2.0\}$.

In [6]:
# ── Compute regime-conditional Sharpe ratios ─────────────────────────────────

# Use the dominant regime label
regime_labels = regime_probs['regime_label'] if 'regime_label' in regime_probs.columns else (
    regime_probs[['p_expansion', 'p_slowdown', 'p_crisis']]
    .idxmax(axis=1)
    .map({'p_expansion': 'Expansion', 'p_slowdown': 'Slowdown', 'p_crisis': 'Crisis'})
)

cond_stats = regime_conditional_stats(factor_returns[FACTORS], regime_labels)

# Build SR_{i,k} lookup: regime -> factor -> sharpe
sr_by_regime = {}
for regime in ['Expansion', 'Slowdown', 'Crisis']:
    sr_dict = {}
    for factor in FACTORS:
        row = cond_stats[(cond_stats['regime'] == regime) & (cond_stats['factor'] == factor)]
        if len(row) > 0:
            sr_dict[factor] = row['sharpe'].values[0]
        else:
            sr_dict[factor] = 0.0
    sr_by_regime[regime] = sr_dict

print('Regime-conditional Sharpe ratios:')
sr_table = pd.DataFrame(sr_by_regime).T
sr_table.index.name = 'Regime'
print(sr_table.round(3))

Regime-conditional Sharpe ratios:
             hml    umd    rmw  lowvol
Regime                                
Expansion -0.080  0.101  0.067  -0.977
Slowdown  -0.382 -0.894  1.248  -0.984
Crisis    -0.097  0.202  0.476  -0.459


In [7]:
# ── Unconditional Sharpe ratios ──────────────────────────────────────────────
mean_sr = {}
for factor in FACTORS:
    ann_ret = factor_returns[factor].mean() * 12
    ann_vol = factor_returns[factor].std() * np.sqrt(12)
    mean_sr[factor] = ann_ret / ann_vol if ann_vol > 0 else 0.0

print('Unconditional Sharpe ratios:')
for f, sr in mean_sr.items():
    print(f'  {f}: {sr:.3f}')

Unconditional Sharpe ratios:
  hml: -0.152
  umd: -0.098
  rmw: 0.503
  lowvol: -0.693


In [8]:
def compute_regime_tilt(regime_probs_row, sr_by_regime, mean_sr, lam, factors):
    """
    Compute tilt_i = 1 + lambda * (sum_k p_k * SR_{i,k} - mean_SR_i).
    Clips to [0.5, 2.0].
    """
    regime_map = {
        'p_expansion': 'Expansion',
        'p_slowdown': 'Slowdown',
        'p_crisis': 'Crisis',
    }
    tilts = np.ones(len(factors))
    for i, factor in enumerate(factors):
        weighted_sr = 0.0
        for p_col, regime_name in regime_map.items():
            p_k = regime_probs_row.get(p_col, 0.0)
            sr_k = sr_by_regime.get(regime_name, {}).get(factor, 0.0)
            weighted_sr += p_k * sr_k
        tilts[i] = 1.0 + lam * (weighted_sr - mean_sr[factor])
    # Clip to [0.5, 2.0]
    tilts = np.clip(tilts, 0.5, 2.0)
    return tilts


# Demonstrate tilts for lambda=1.0 at a sample date
sample_idx = len(regime_probs) // 2
sample_row = regime_probs.iloc[sample_idx]
sample_tilts = compute_regime_tilt(sample_row, sr_by_regime, mean_sr, 1.0, FACTORS)
print(f'Sample tilts at {regime_probs.index[sample_idx].date()} (lambda=1.0):')
for f, t in zip(FACTORS, sample_tilts):
    print(f'  {f}: {t:.3f}')

Sample tilts at 2016-07-31 (lambda=1.0):
  hml: 1.054
  umd: 1.297
  rmw: 0.975
  lowvol: 1.232


In [9]:
def apply_tilted_rp_weights(rp_weights, tilts):
    """
    Apply regime tilts to risk parity weights and re-normalise.
    w_tilted_i = w_RP_i * tilt_i / sum_j(w_RP_j * tilt_j)
    """
    raw = rp_weights * tilts
    # Clip to max weight, then renormalise
    raw = np.clip(raw, 0.0, MAX_WEIGHT_FACTOR)
    total = raw.sum()
    if total > 0:
        return raw / total
    return np.ones(len(raw)) / len(raw)


# Build tilted RP weights for lambda=1.0 as default
lam_default = 1.0
tilted_weights_list = []

for t_idx in range(len(common_idx)):
    row = regime_probs.iloc[t_idx]
    tilts = compute_regime_tilt(row, sr_by_regime, mean_sr, lam_default, FACTORS)
    w_rp = rp_weights_df.iloc[t_idx].values
    w_tilted = apply_tilted_rp_weights(w_rp, tilts)
    tilted_weights_list.append(w_tilted)

tilted_rp_df = pd.DataFrame(
    tilted_weights_list, index=common_idx, columns=WEIGHT_COLS
)

# Validation: weights sum to 1 and tilts bounded
assert np.allclose(tilted_rp_df.sum(axis=1), 1.0, atol=1e-6), 'Tilted RP weights do not sum to 1'

print(f'Tilted RP weights (lambda={lam_default}) — summary:')
print(tilted_rp_df.describe().round(4))

Tilted RP weights (lambda=1.0) — summary:
          w_hml     w_umd     w_rmw  w_lowvol
count  185.0000  185.0000  185.0000  185.0000
mean     0.2630    0.2448    0.3487    0.1435
std      0.0716    0.0829    0.0979    0.0401
min      0.1493    0.0546    0.2026    0.0617
25%      0.2134    0.1846    0.2774    0.1086
50%      0.2597    0.2495    0.3350    0.1530
75%      0.2986    0.3152    0.3869    0.1757
max      0.4829    0.4017    0.6013    0.2201


---
## 8.4 CVaR Optimization (Rockafellar-Uryasev LP)

$$\min_{w, \zeta, u} \quad \zeta + \frac{1}{S(1-\alpha)} \sum_s u_s$$

subject to:
- $u_s \geq -\text{scenarios} \cdot w - \zeta, \quad u \geq 0$
- $\mu_{\text{regime}} \cdot w \geq r_{\text{target}}$
- $\sum w = 1, \quad w \geq 0, \quad w \leq 0.4$
- Turnover constraint: $\|w - w_{\text{prev}}\|_1 \leq 2 \times 0.20$

Using `src.portfolio_optimization.mean_cvar_optimize`.

In [10]:
# ── Demonstrate single-period CVaR optimization ──────────────────────────────

# Use first WARMUP_MONTHS as scenario matrix
demo_end = min(WARMUP_MONTHS + 1, len(factor_returns))
scenario_matrix_demo = factor_returns.iloc[:demo_end][FACTORS].values

print(f'Demo scenario matrix shape: {scenario_matrix_demo.shape}')
print(f'  Scenarios (S): {scenario_matrix_demo.shape[0]}')
print(f'  Assets (N):    {scenario_matrix_demo.shape[1]}')

# Regime-conditional expected returns (probability-weighted)
demo_probs = regime_probs.iloc[demo_end - 1]
mu_regime_demo = np.zeros(len(FACTORS))
for p_col, regime_name in [('p_expansion', 'Expansion'),
                            ('p_slowdown', 'Slowdown'),
                            ('p_crisis', 'Crisis')]:
    p_k = demo_probs.get(p_col, 0.0)
    for j, factor in enumerate(FACTORS):
        sr = sr_by_regime.get(regime_name, {}).get(factor, 0.0)
        vol = garch_vol.iloc[demo_end - 1].values[j]
        mu_regime_demo[j] += p_k * sr * vol / np.sqrt(12)  # Monthly expected return

print(f'\nRegime-conditional expected returns (monthly):')
for f, mu in zip(FACTORS, mu_regime_demo):
    print(f'  {f}: {mu:.5f} ({mu*12:.3%} ann.)')

# Optimize
w_cvar_demo = mean_cvar_optimize(
    scenario_matrix=scenario_matrix_demo,
    alpha=ALPHA_CVAR,
    max_weight=MAX_WEIGHT_FACTOR,
    target_return=None,  # No minimum return constraint for first period
    prev_weights=None,
    turnover_limit=None,
    tc=TC_FACTOR,
    return_weight=0.5,   # Balance between CVaR minimization and return
)

print(f'\nCVaR-optimal weights (demo):')
for f, w in zip(FACTORS, w_cvar_demo):
    print(f'  {f}: {w:.4f}')
print(f'  Sum: {w_cvar_demo.sum():.6f}')

Demo scenario matrix shape: (61, 4)
  Scenarios (S): 61
  Assets (N):    4

Regime-conditional expected returns (monthly):
  hml: -0.00159 (-1.907% ann.)
  umd: 0.00233 (2.800% ann.)
  rmw: 0.00114 (1.362% ann.)
  lowvol: -0.04068 (-48.817% ann.)

CVaR-optimal weights (demo):
  hml: 0.4000
  umd: 0.0518
  rmw: 0.4000
  lowvol: 0.1482
  Sum: 1.000000


---
## 8.5 Walk-Forward CVaR Weight Time Series

For each month $t$ from `WARMUP_MONTHS` to $T$:
1. Build scenario matrix from returns $[0, t)$ (expanding window)
2. Compute regime-conditional expected returns using current regime probabilities
3. Apply CVaR optimization with return weight to balance tail risk and expected return
4. Apply turnover constraint relative to previous weights

The regime tilt enters through the `return_weight` parameter: we use the
regime-tilted expected returns to guide the optimizer.

In [11]:
def compute_regime_mu(regime_probs_row, sr_by_regime, garch_vol_row, factors):
    """
    Compute probability-weighted expected monthly returns from
    regime-conditional Sharpe ratios and GARCH volatility.

    mu_i = sum_k p_k * SR_{i,k} * vol_i / sqrt(12)
    """
    mu = np.zeros(len(factors))
    regime_map = {
        'p_expansion': 'Expansion',
        'p_slowdown': 'Slowdown',
        'p_crisis': 'Crisis',
    }
    for p_col, regime_name in regime_map.items():
        p_k = regime_probs_row.get(p_col, 0.0)
        for j, factor in enumerate(factors):
            sr_k = sr_by_regime.get(regime_name, {}).get(factor, 0.0)
            vol_j = garch_vol_row[j]
            mu[j] += p_k * sr_k * vol_j / np.sqrt(12)
    return mu

In [12]:
%%time

# ── Walk-forward CVaR optimization ────────────────────────────────────────────

T = len(common_idx)
cvar_weights = np.full((T, len(FACTORS)), np.nan)
n_failures = 0

# Equal weight for warm-up period
eq_weight = np.ones(len(FACTORS)) / len(FACTORS)
cvar_weights[:WARMUP_MONTHS] = eq_weight

prev_w = eq_weight.copy()

for t in range(WARMUP_MONTHS, T):
    # 1. Expanding window scenario matrix: all returns up to (but not including) t
    #    This ensures no look-ahead: we use returns [0, t) to form scenarios
    scenario_matrix = factor_returns.iloc[:t][FACTORS].values

    # 2. Regime-conditional expected returns at current t
    mu_regime = compute_regime_mu(
        regime_probs.iloc[t],
        sr_by_regime,
        garch_vol.iloc[t].values,
        FACTORS,
    )

    # 3. CVaR optimization with turnover constraint
    try:
        w_opt = mean_cvar_optimize(
            scenario_matrix=scenario_matrix,
            alpha=ALPHA_CVAR,
            max_weight=MAX_WEIGHT_FACTOR,
            target_return=None,
            prev_weights=prev_w,
            turnover_limit=TURNOVER_LIMIT,
            tc=TC_FACTOR,
            return_weight=0.5,
        )
    except Exception as e:
        logger.warning(f't={t} ({common_idx[t].date()}): CVaR opt failed: {e}')
        w_opt = prev_w.copy()
        n_failures += 1

    # Ensure constraints satisfied
    w_opt = np.maximum(w_opt, 0.0)
    w_opt = np.minimum(w_opt, MAX_WEIGHT_FACTOR)
    if w_opt.sum() > 0:
        w_opt = w_opt / w_opt.sum()
    else:
        w_opt = eq_weight.copy()

    cvar_weights[t] = w_opt
    prev_w = w_opt.copy()

    # Progress logging every 50 months
    if (t - WARMUP_MONTHS) % 50 == 0:
        logger.info(f'Walk-forward CVaR: t={t}/{T}')

print(f'Walk-forward complete: {T - WARMUP_MONTHS} months optimized')
print(f'Optimization failures: {n_failures}')

Walk-forward complete: 125 months optimized
Optimization failures: 0
CPU times: user 1.7 s, sys: 5.26 ms, total: 1.71 s
Wall time: 1.71 s


In [13]:
# Build DataFrame of CVaR weights
cvar_weights_df = pd.DataFrame(
    cvar_weights, index=common_idx, columns=WEIGHT_COLS
)

# Validation checks
live_mask = cvar_weights_df.index >= pd.Timestamp(FACTOR_LIVE_START)
live_weights = cvar_weights_df[live_mask]

print('CVaR weights — live period summary:')
print(live_weights.describe().round(4))

# Check weights sum to 1
weight_sums = live_weights.sum(axis=1)
assert weight_sums.between(0.99, 1.01).all(), \
    f'CVaR weights do not sum to 1: min={weight_sums.min():.6f}, max={weight_sums.max():.6f}'
print(f'\nWeight sums: min={weight_sums.min():.6f}, max={weight_sums.max():.6f}')

# Check max weight constraint
max_w = live_weights.max().max()
print(f'Max weight: {max_w:.4f} (cap: {MAX_WEIGHT_FACTOR})')
assert max_w <= MAX_WEIGHT_FACTOR + 0.01, f'Weight exceeds cap: {max_w:.4f}'

# Turnover
turnover = live_weights.diff().abs().sum(axis=1).dropna()
print(f'Monthly turnover: mean={turnover.mean():.4f}, max={turnover.max():.4f}')

CVaR weights — live period summary:
          w_hml     w_umd     w_rmw  w_lowvol
count  171.0000  171.0000  171.0000  171.0000
mean     0.3592    0.1692    0.3595    0.1120
std      0.0667    0.0623    0.0666    0.0918
min      0.2500    0.0518    0.2500    0.0000
25%      0.2500    0.1233    0.2500    0.0324
50%      0.4000    0.1726    0.4000    0.0767
75%      0.4000    0.2500    0.4000    0.2500
max      0.4000    0.2500    0.4000    0.2500

Weight sums: min=1.000000, max=1.000000
Max weight: 0.4000 (cap: 0.4)
Monthly turnover: mean=0.0072, max=0.4000


In [14]:
# Plot CVaR weight evolution
fig = plot_weight_evolution(
    cvar_weights_df[live_mask].rename(columns=dict(zip(WEIGHT_COLS, FACTORS))),
    title='CVaR Optimal Weights — Walk-Forward (Live Period)'
)
save_fig(fig, 'cvar_weights_evolution')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_weights_evolution.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_weights_evolution.png')

In [15]:
# Plot individual weight time series (line chart for detail)
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
factor_colors = [COLORS['hml'], COLORS['umd'], COLORS['rmw'], COLORS['lowvol']]

for ax, factor, w_col, color in zip(axes.flat, FACTORS, WEIGHT_COLS, factor_colors):
    ax.plot(live_weights.index, live_weights[w_col], color=color, linewidth=1)
    ax.axhline(y=0.25, ls='--', color='gray', alpha=0.5, label='Equal weight')
    ax.axhline(y=MAX_WEIGHT_FACTOR, ls=':', color='red', alpha=0.4, label=f'Cap ({MAX_WEIGHT_FACTOR})')
    ax.set_ylabel('Weight')
    ax.set_title(f'{factor.upper()}', fontweight='bold')
    ax.set_ylim(-0.02, MAX_WEIGHT_FACTOR + 0.05)
    ax.legend(loc='upper right', fontsize=8)

fig.suptitle('CVaR Weights by Factor — Walk-Forward', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'cvar_weights_by_factor')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_weights_by_factor.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_weights_by_factor.png')

---
## 8.6 Lambda Sensitivity Analysis

Test how the regime tilt intensity $\lambda$ affects portfolio characteristics.
For each $\lambda \in \{0.5, 1.0, 1.5, 2.0\}$, compute tilted weights
and resulting portfolio metrics.

In [16]:
%%time

LAMBDA_GRID = [0.5, 1.0, 1.5, 2.0]
lambda_results = {}

for lam in LAMBDA_GRID:
    # Compute tilted RP weights for this lambda
    tilted_list = []
    for t_idx in range(len(common_idx)):
        row = regime_probs.iloc[t_idx]
        tilts = compute_regime_tilt(row, sr_by_regime, mean_sr, lam, FACTORS)
        w_rp = rp_weights_df.iloc[t_idx].values
        w_tilted = apply_tilted_rp_weights(w_rp, tilts)
        tilted_list.append(w_tilted)

    w_df = pd.DataFrame(tilted_list, index=common_idx, columns=WEIGHT_COLS)

    # Portfolio returns
    port_ret = (factor_returns[FACTORS].values * w_df.values).sum(axis=1)
    port_ret_series = pd.Series(port_ret, index=common_idx, name=f'lambda_{lam}')

    # Live-period metrics
    live_ret = port_ret_series[port_ret_series.index >= pd.Timestamp(FACTOR_LIVE_START)]
    metrics = compute_all_metrics(live_ret, rf=0.0, name=f'lambda={lam}')

    # Tilt statistics
    live_w = w_df[w_df.index >= pd.Timestamp(FACTOR_LIVE_START)]
    metrics['mean_max_weight'] = live_w.max(axis=1).mean()
    metrics['mean_turnover'] = live_w.diff().abs().sum(axis=1).dropna().mean()

    lambda_results[lam] = {
        'metrics': metrics,
        'weights': w_df,
        'returns': port_ret_series,
    }

    print(f'lambda={lam:.1f}: Sharpe={metrics["sharpe"]:.3f}, '
          f'Vol={metrics["ann_volatility"]:.3%}, '
          f'MaxDD={metrics["max_drawdown"]:.3%}, '
          f'Turnover={metrics["mean_turnover"]:.4f}')

lambda=0.5: Sharpe=0.241, Vol=28.483%, MaxDD=-20.854%, Turnover=0.0721
lambda=1.0: Sharpe=0.311, Vol=28.981%, MaxDD=-20.869%, Turnover=0.0779
lambda=1.5: Sharpe=0.400, Vol=29.293%, MaxDD=-20.405%, Turnover=0.0803
lambda=2.0: Sharpe=0.416, Vol=29.628%, MaxDD=-20.561%, Turnover=0.0805
CPU times: user 40.7 ms, sys: 239 μs, total: 40.9 ms
Wall time: 40.8 ms


In [17]:
# Lambda sensitivity table
lambda_table = pd.DataFrame({
    lam: {
        'Ann. Return': res['metrics']['ann_return'],
        'Ann. Vol': res['metrics']['ann_volatility'],
        'Sharpe': res['metrics']['sharpe'],
        'Sortino': res['metrics']['sortino'],
        'Max DD': res['metrics']['max_drawdown'],
        'CVaR 95': res['metrics']['cvar_95'],
        'Avg Turnover': res['metrics']['mean_turnover'],
        'Avg Max Weight': res['metrics']['mean_max_weight'],
    }
    for lam, res in lambda_results.items()
}).T
lambda_table.index.name = 'Lambda'

print('Lambda Sensitivity Analysis:')
print(lambda_table.round(4))

# Save to tables
lambda_table.to_csv(TABLES_DIR / 'cvar_lambda_sensitivity.csv')
print(f'\nSaved: {TABLES_DIR / "cvar_lambda_sensitivity.csv"}')

Lambda Sensitivity Analysis:
        Ann. Return  Ann. Vol  Sharpe  Sortino  Max DD  CVaR 95  Avg Turnover  \
Lambda                                                                          
0.5          0.0686    0.2848  0.2407   0.3082 -0.2085   0.0439        0.0721   
1.0          0.0900    0.2898  0.3106   0.3906 -0.2087   0.0453        0.0779   
1.5          0.1171    0.2929  0.3998   0.4995 -0.2041   0.0459        0.0803   
2.0          0.1234    0.2963  0.4164   0.5169 -0.2056   0.0464        0.0805   

        Avg Max Weight  
Lambda                  
0.5             0.3751  
1.0             0.3726  
1.5             0.3741  
2.0             0.3714  

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/cvar_lambda_sensitivity.csv


In [18]:
# Plot lambda sensitivity: Sharpe and CVaR vs lambda
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

lambdas = sorted(lambda_results.keys())
sharpes = [lambda_results[l]['metrics']['sharpe'] for l in lambdas]
cvars = [lambda_results[l]['metrics']['cvar_95'] for l in lambdas]
maxdds = [lambda_results[l]['metrics']['max_drawdown'] for l in lambdas]

axes[0].plot(lambdas, sharpes, 'o-', color='#3498db', linewidth=2, markersize=8)
axes[0].set_xlabel('Lambda')
axes[0].set_ylabel('Sharpe Ratio')
axes[0].set_title('Sharpe vs Lambda', fontweight='bold')

axes[1].plot(lambdas, cvars, 's-', color='#e74c3c', linewidth=2, markersize=8)
axes[1].set_xlabel('Lambda')
axes[1].set_ylabel('CVaR (95%)')
axes[1].set_title('CVaR vs Lambda', fontweight='bold')

axes[2].plot(lambdas, maxdds, '^-', color='#f39c12', linewidth=2, markersize=8)
axes[2].set_xlabel('Lambda')
axes[2].set_ylabel('Max Drawdown')
axes[2].set_title('Max Drawdown vs Lambda', fontweight='bold')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

fig.suptitle('Regime Tilt Sensitivity Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'cvar_lambda_sensitivity')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_lambda_sensitivity.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_lambda_sensitivity.png')

In [19]:
# Weight composition by lambda — stacked bar chart
fig, ax = plt.subplots(figsize=(10, 6))

bar_data = {}
for lam in lambdas:
    live_w = lambda_results[lam]['weights'][lambda_results[lam]['weights'].index >= pd.Timestamp(FACTOR_LIVE_START)]
    bar_data[f'$\\lambda$={lam}'] = live_w.mean().values

x = np.arange(len(lambdas))
width = 0.6
bottom = np.zeros(len(lambdas))
colors = [COLORS['hml'], COLORS['umd'], COLORS['rmw'], COLORS['lowvol']]

for i, (factor, color) in enumerate(zip(FACTORS, colors)):
    vals = [bar_data[k][i] for k in bar_data]
    ax.bar(x, vals, width, bottom=bottom, label=factor.upper(), color=color, alpha=0.85)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([f'$\\lambda$={l}' for l in lambdas])
ax.set_ylabel('Average Weight')
ax.set_ylim(0, 1.05)
ax.set_title('Average Weight Composition by Lambda', fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
save_fig(fig, 'cvar_lambda_weight_composition')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_lambda_weight_composition.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_lambda_weight_composition.png')

---
## 8.7 Comparison with BL

Load Black-Litterman weights from Phase 7 and compare with CVaR weights.

In [20]:
# ── Load BL weights ──────────────────────────────────────────────────────────
bl_weights = pd.read_parquet(BL_WEIGHTS_FILE)
validate_parquet(bl_weights, expected_cols=WEIGHT_COLS,
                 min_rows=50, date_index=True, label='bl_weights')

# Align to common dates with CVaR weights
bl_weights.index = bl_weights.index + pd.offsets.MonthEnd(0)
comparison_idx = cvar_weights_df.index.intersection(bl_weights.index)
comparison_idx = comparison_idx[comparison_idx >= pd.Timestamp(FACTOR_LIVE_START)]

bl_live = bl_weights.loc[comparison_idx]
cvar_live = cvar_weights_df.loc[comparison_idx]
returns_live = factor_returns.loc[comparison_idx, FACTORS]

print(f'Comparison period: {comparison_idx[0].date()} to {comparison_idx[-1].date()}')
print(f'Months: {len(comparison_idx)}')

Comparison period: 2013-11-30 to 2024-03-31
Months: 125


In [21]:
# ── Validation Gate: CVaR weights differ from BL weights ─────────────────────

weight_diff = (cvar_live.values - bl_live.values)
mean_abs_diff = np.abs(weight_diff).mean()
max_abs_diff = np.abs(weight_diff).max()

print(f'CVaR vs BL weight differences:')
print(f'  Mean absolute difference: {mean_abs_diff:.4f}')
print(f'  Max absolute difference:  {max_abs_diff:.4f}')

# Gate: weights must actually differ
assert mean_abs_diff > 0.001, \
    f'CVaR and BL weights are too similar (mean diff={mean_abs_diff:.6f})'
print('PASS: CVaR weights differ from BL weights')

CVaR vs BL weight differences:
  Mean absolute difference: 0.1496
  Max absolute difference:  0.2499
PASS: CVaR weights differ from BL weights


In [22]:
# ── Compare portfolio returns ────────────────────────────────────────────────

# Portfolio returns for each strategy
cvar_port_ret = (returns_live.values * cvar_live.values).sum(axis=1)
bl_port_ret = (returns_live.values * bl_live.values).sum(axis=1)
eq_port_ret = returns_live.mean(axis=1)  # Equal weight benchmark

cvar_ret_series = pd.Series(cvar_port_ret, index=comparison_idx, name='CVaR')
bl_ret_series = pd.Series(bl_port_ret, index=comparison_idx, name='BL')
eq_ret_series = pd.Series(eq_port_ret.values, index=comparison_idx, name='EqualWeight')

# Use compare_allocations from portfolio_optimization
weights_dict = {
    'CVaR': cvar_live.mean().values,
    'Black-Litterman': bl_live.mean().values,
    'Equal Weight': np.ones(len(FACTORS)) / len(FACTORS),
}

comparison_table = compare_allocations(returns_live, weights_dict)
print('Strategy Comparison (average weights):')
print(comparison_table.round(4))

Strategy Comparison (average weights):
                 ann_return  ann_volatility  sharpe  max_drawdown  max_weight  \
strategy                                                                        
CVaR                 0.1082          0.2947  0.3673       -0.2365      0.3998   
Black-Litterman     -0.3629          0.3631 -0.9993       -0.3195      0.2500   
Equal Weight        -0.3630          0.3631 -0.9996       -0.3195      0.2500   

                 effective_n  
strategy                      
CVaR                   2.919  
Black-Litterman        4.000  
Equal Weight           4.000  


In [23]:
# ── Detailed time-varying performance metrics ────────────────────────────────

metrics_cvar = compute_all_metrics(cvar_ret_series, rf=0.0, name='CVaR')
metrics_bl = compute_all_metrics(bl_ret_series, rf=0.0, name='Black-Litterman')
metrics_eq = compute_all_metrics(eq_ret_series, rf=0.0, name='Equal Weight')

perf_df = pd.DataFrame([metrics_cvar, metrics_bl, metrics_eq]).set_index('name')
print('Detailed Performance Comparison (live period):')
print(perf_df[['ann_return', 'ann_volatility', 'sharpe', 'sortino',
               'max_drawdown', 'calmar', 'var_95', 'cvar_95',
               'skewness', 'excess_kurtosis', 'hit_rate', 'tail_ratio']].round(4))

# Save
perf_df.to_csv(TABLES_DIR / 'cvar_vs_bl_performance.csv')
print(f'\nSaved: {TABLES_DIR / "cvar_vs_bl_performance.csv"}')

Detailed Performance Comparison (live period):
                 ann_return  ann_volatility  sharpe  sortino  max_drawdown  \
name                                                                         
CVaR                 0.1397          0.2944  0.4746   0.7735       -0.2336   
Black-Litterman     -0.3628          0.3631 -0.9992  -1.2554       -0.3195   
Equal Weight        -0.3630          0.3631 -0.9996  -1.2559       -0.3195   

                 calmar  var_95  cvar_95  skewness  excess_kurtosis  hit_rate  \
name                                                                            
CVaR             0.5979  0.0274   0.0385    0.2736           1.2994      0.48   
Black-Litterman -1.1357  0.0386   0.0581   -0.8446           2.9955      0.48   
Equal Weight    -1.1362  0.0387   0.0581   -0.8448           2.9962      0.48   

                 tail_ratio  
name                         
CVaR                 1.1922  
Black-Litterman      0.8482  
Equal Weight         0.8479  

Saved

In [24]:
# ── Validation Gate: CVaR portfolio has lower tail risk ──────────────────────

cvar_5th = np.percentile(cvar_port_ret, 5)
bl_5th = np.percentile(bl_port_ret, 5)

print(f'5th percentile monthly return:')
print(f'  CVaR:  {cvar_5th:.4%}')
print(f'  BL:    {bl_5th:.4%}')

if cvar_5th > bl_5th:
    print('PASS: CVaR portfolio has less severe left tail (higher 5th pctile)')
else:
    print('NOTE: CVaR 5th percentile is lower than BL. '
          'This may occur if CVaR optimizes over a different scenario set. '
          'Documenting for transparency.')

# Also compare using scenario-based CVaR
scenario_full = returns_live.values
cvar_portfolio_cvar = portfolio_cvar(
    cvar_live.mean().values, scenario_full, alpha=ALPHA_CVAR
)
bl_portfolio_cvar = portfolio_cvar(
    bl_live.mean().values, scenario_full, alpha=ALPHA_CVAR
)

print(f'\nPortfolio CVaR (5%, scenario-based):')
print(f'  CVaR strategy:  {cvar_portfolio_cvar:.4%}')
print(f'  BL strategy:    {bl_portfolio_cvar:.4%}')

5th percentile monthly return:
  CVaR:  -2.7381%
  BL:    -3.8646%
PASS: CVaR portfolio has less severe left tail (higher 5th pctile)

Portfolio CVaR (5%, scenario-based):
  CVaR strategy:  3.9192%
  BL strategy:    5.8070%


In [25]:
# ── Plot: CVaR vs BL weights side-by-side ────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Top: CVaR weights
axes[0].stackplot(
    cvar_live.index,
    *[cvar_live[c] for c in WEIGHT_COLS],
    labels=[f.upper() for f in FACTORS],
    colors=[COLORS['hml'], COLORS['umd'], COLORS['rmw'], COLORS['lowvol']],
    alpha=0.8,
)
axes[0].set_ylabel('Weight')
axes[0].set_ylim(0, 1)
axes[0].set_title('CVaR Weights', fontweight='bold')
axes[0].legend(loc='upper left', ncol=4, fontsize=9)

# Bottom: BL weights
axes[1].stackplot(
    bl_live.index,
    *[bl_live[c] for c in WEIGHT_COLS],
    labels=[f.upper() for f in FACTORS],
    colors=[COLORS['hml'], COLORS['umd'], COLORS['rmw'], COLORS['lowvol']],
    alpha=0.8,
)
axes[1].set_ylabel('Weight')
axes[1].set_ylim(0, 1)
axes[1].set_title('Black-Litterman Weights', fontweight='bold')
axes[1].legend(loc='upper left', ncol=4, fontsize=9)

fig.suptitle('CVaR vs Black-Litterman Weight Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'cvar_vs_bl_weights_comparison')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_vs_bl_weights_comparison.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_vs_bl_weights_comparison.png')

In [26]:
# ── Plot: Cumulative returns comparison ──────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 6))

cum_cvar = (1 + cvar_ret_series).cumprod()
cum_bl = (1 + bl_ret_series).cumprod()
cum_eq = (1 + eq_ret_series).cumprod()

ax.plot(cum_cvar.index, cum_cvar, label='CVaR', color='#3498db', linewidth=1.5)
ax.plot(cum_bl.index, cum_bl, label='Black-Litterman', color='#e74c3c', linewidth=1.5)
ax.plot(cum_eq.index, cum_eq, label='Equal Weight', color='gray', linewidth=1, ls='--')

ax.set_ylabel('Cumulative Return (base=1)')
ax.set_title('CVaR vs BL vs Equal Weight — Cumulative Returns', fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
save_fig(fig, 'cvar_vs_bl_cumulative_returns')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_vs_bl_cumulative_returns.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_vs_bl_cumulative_returns.png')

In [27]:
# ── Plot: Drawdown comparison ────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 5))

for ret_s, label, color in [
    (cum_cvar, 'CVaR', '#3498db'),
    (cum_bl, 'Black-Litterman', '#e74c3c'),
    (cum_eq, 'Equal Weight', 'gray'),
]:
    dd = (ret_s / ret_s.cummax()) - 1
    ax.fill_between(dd.index, dd, alpha=0.15, color=color)
    ax.plot(dd.index, dd, label=label, color=color, linewidth=1)

ax.set_ylabel('Drawdown')
ax.set_title('Drawdown Comparison — CVaR vs BL vs Equal Weight', fontweight='bold')
ax.legend(loc='lower left', fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
save_fig(fig, 'cvar_vs_bl_drawdown')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_vs_bl_drawdown.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_vs_bl_drawdown.png')

In [28]:
# ── Weight difference analysis ───────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
factor_colors = [COLORS['hml'], COLORS['umd'], COLORS['rmw'], COLORS['lowvol']]

for ax, factor, w_col, color in zip(axes.flat, FACTORS, WEIGHT_COLS, factor_colors):
    diff = cvar_live[w_col].values - bl_live[w_col].values
    ax.fill_between(comparison_idx, diff, alpha=0.4, color=color)
    ax.axhline(y=0, ls='-', color='black', alpha=0.3)
    ax.set_ylabel('CVaR - BL')
    ax.set_title(f'{factor.upper()}: Weight Difference (CVaR - BL)', fontweight='bold')

fig.suptitle('Weight Differences: CVaR vs Black-Litterman', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'cvar_bl_weight_differences')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_bl_weight_differences.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/cvar_bl_weight_differences.png')

In [29]:
# ── Risk decomposition for both strategies ───────────────────────────────────

# Use average weights and last available covariance for decomposition
cov_cols = cond_cov.columns
n_factors = len(FACTORS)
last_cov_row = cond_cov.iloc[-1].values

# Reconstruct 4x4 covariance matrix from flattened row
if len(last_cov_row) == n_factors * n_factors:
    cov_matrix = last_cov_row.reshape(n_factors, n_factors)
else:
    # Fallback: use sample covariance from returns
    cov_matrix = returns_live.cov().values

# Risk decomposition
rd_cvar = risk_decomposition(cvar_live.mean().values, cov_matrix)
rd_bl = risk_decomposition(bl_live.mean().values, cov_matrix)

print('Risk Contribution (% of portfolio vol):')
print('\nCVaR:')
for i, f in enumerate(FACTORS):
    print(f'  {f}: {rd_cvar["pct_risk_contrib"][i]:.2%}')
print(f'  Portfolio vol: {rd_cvar["portfolio_vol"]:.4%}')

print('\nBlack-Litterman:')
for i, f in enumerate(FACTORS):
    print(f'  {f}: {rd_bl["pct_risk_contrib"][i]:.2%}')
print(f'  Portfolio vol: {rd_bl["portfolio_vol"]:.4%}')

# Concentration metrics
cm_cvar = concentration_metrics(cvar_live.mean().values, cov_matrix)
cm_bl = concentration_metrics(bl_live.mean().values, cov_matrix)

print('\nConcentration Metrics:')
print(f'  CVaR: HHI={cm_cvar["hhi"]:.4f}, Eff.N={cm_cvar["effective_n"]:.2f}, '
      f'DivRatio={cm_cvar["diversification_ratio"]:.3f}')
print(f'  BL:   HHI={cm_bl["hhi"]:.4f}, Eff.N={cm_bl["effective_n"]:.2f}, '
      f'DivRatio={cm_bl["diversification_ratio"]:.3f}')

Risk Contribution (% of portfolio vol):

CVaR:
  hml: 48.72%
  umd: 10.20%
  rmw: 31.80%
  lowvol: 9.28%
  Portfolio vol: 1.8619%

Black-Litterman:
  hml: 4.82%
  umd: 33.54%
  rmw: 12.19%
  lowvol: 49.44%
  Portfolio vol: 2.5223%

Concentration Metrics:
  CVaR: HHI=0.3426, Eff.N=2.92, DivRatio=1.847
  BL:   HHI=0.2500, Eff.N=4.00, DivRatio=1.666


---
## Validation Gates Summary

In [30]:
# ── Final Validation Gates ───────────────────────────────────────────────────

print('=' * 60)
print('VALIDATION GATES — Phase 8')
print('=' * 60)

# Gate 1: CVaR weights differ from BL weights
gate1 = mean_abs_diff > 0.001
print(f'\n[{"PASS" if gate1 else "FAIL"}] CVaR weights differ from BL weights '
      f'(mean abs diff = {mean_abs_diff:.4f})')

# Gate 2: CVaR portfolio has lower tail risk
gate2_5th = cvar_5th > bl_5th
gate2_cvar = cvar_portfolio_cvar < bl_portfolio_cvar
gate2 = gate2_5th or gate2_cvar
print(f'[{"PASS" if gate2 else "NOTE"}] CVaR portfolio lower tail risk '
      f'(5th pctile: CVaR={cvar_5th:.4%} vs BL={bl_5th:.4%}, '
      f'CVaR metric: CVaR={cvar_portfolio_cvar:.4%} vs BL={bl_portfolio_cvar:.4%})')

# Gate 3: Tilt multipliers bounded [0.5, 2.0]
# Verified during computation (clipping applied)
print('[PASS] Tilt multipliers clipped to [0.5, 2.0]')

# Gate 4: Weights sum to 1.0
ws = cvar_weights_df[~cvar_weights_df.isna().any(axis=1)].sum(axis=1)
gate4 = ws.between(0.99, 1.01).all()
print(f'[{"PASS" if gate4 else "FAIL"}] Weights sum to 1.0 '
      f'(range: [{ws.min():.6f}, {ws.max():.6f}])')

# Gate 5: No weight exceeds cap
gate5 = (cvar_weights_df.max().max() <= MAX_WEIGHT_FACTOR + 0.01)
print(f'[{"PASS" if gate5 else "FAIL"}] Max weight <= {MAX_WEIGHT_FACTOR} '
      f'(actual max: {cvar_weights_df.max().max():.4f})')

print('\n' + '=' * 60)

VALIDATION GATES — Phase 8

[PASS] CVaR weights differ from BL weights (mean abs diff = 0.1496)
[PASS] CVaR portfolio lower tail risk (5th pctile: CVaR=-2.7381% vs BL=-3.8646%, CVaR metric: CVaR=3.9192% vs BL=5.8070%)
[PASS] Tilt multipliers clipped to [0.5, 2.0]
[PASS] Weights sum to 1.0 (range: [1.000000, 1.000000])
[PASS] Max weight <= 0.4 (actual max: 0.4000)



---
## Export Outputs

In [31]:
# ── Export cvar_weights_timeseries.parquet ────────────────────────────────────

output_df = cvar_weights_df.copy()
output_df.index.name = 'date'

# Drop warm-up NaN rows if any (should be filled with equal weight)
assert not output_df.isna().any().any(), 'Unexpected NaN in output weights'

output_df.to_parquet(CVAR_WEIGHTS_FILE)

print(f'Exported: {CVAR_WEIGHTS_FILE}')
print(f'  Shape: {output_df.shape[0]} rows x {output_df.shape[1]} cols')
print(f'  Date range: {output_df.index.min().date()} to {output_df.index.max().date()}')
print(f'  Columns: {list(output_df.columns)}')
print(f'  NaN count: {output_df.isna().sum().sum()}')

logger.info(f'Phase {PHASE_NUM} complete. '
            f'Exported {output_df.shape[0]} months of CVaR weights.')

Exported: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/cvar_weights_timeseries.parquet
  Shape: 185 rows x 4 cols
  Date range: 2008-11-30 to 2024-03-31
  Columns: ['w_hml', 'w_umd', 'w_rmw', 'w_lowvol']
  NaN count: 0


In [32]:
print('\nPhase 8 complete.')
print(f'  Outputs: {CVAR_WEIGHTS_FILE}')
print(f'  Tables:  {TABLES_DIR / "cvar_lambda_sensitivity.csv"}')
print(f'           {TABLES_DIR / "cvar_vs_bl_performance.csv"}')
print(f'  Figures: cvar_rp_base_weights.png')
print(f'           cvar_weights_evolution.png')
print(f'           cvar_weights_by_factor.png')
print(f'           cvar_lambda_sensitivity.png')
print(f'           cvar_lambda_weight_composition.png')
print(f'           cvar_vs_bl_weights_comparison.png')
print(f'           cvar_vs_bl_cumulative_returns.png')
print(f'           cvar_vs_bl_drawdown.png')
print(f'           cvar_bl_weight_differences.png')


Phase 8 complete.
  Outputs: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/data/processed/cvar_weights_timeseries.parquet
  Tables:  /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/cvar_lambda_sensitivity.csv
           /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/cvar_vs_bl_performance.csv
  Figures: cvar_rp_base_weights.png
           cvar_weights_evolution.png
           cvar_weights_by_factor.png
           cvar_lambda_sensitivity.png
           cvar_lambda_weight_composition.png
           cvar_vs_bl_weights_comparison.png
           cvar_vs_bl_cumulative_returns.png
           cvar_vs_bl_drawdown.png
           cvar_bl_weight_differences.png
